# EDA inicial: cancelaciones hoteleras

Análisis exploratorio del dataset de reservas hoteleras. El objetivo es entender los datos
y justificar las decisiones de preprocesado y modelado que se aplican en `src/`:

1. Vista general del dataset
2. Variable objetivo y balance de clases
3. Calidad de los datos: nulos, duplicados y valores anómalos
4. Fuga de información
5. Variables numéricas
6. Variables categóricas
7. Relación de las variables con la cancelación
8. Evolución temporal
9. Conclusiones para el modelado

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (10, 4)

# Permite ejecutar el notebook tanto desde la raíz del repositorio como desde notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_PATH = ROOT / "data" / "raw" / "dataset_practica_final.csv"
TARGET = "is_canceled"

df = pd.read_csv(DATA_PATH)
df.shape

## 1. Vista general

In [ ]:
df.head()

In [ ]:
df.info()

## 2. Variable objetivo

Un desbalance moderado condiciona la métrica: con clases desiguales, la accuracy de un
clasificador trivial que siempre predice la clase mayoritaria ya es alta.

In [ ]:
balance = pd.DataFrame({
    "n": df[TARGET].value_counts(),
    "%": df[TARGET].value_counts(normalize=True).mul(100).round(2),
})
print(balance)
print(f"\nAccuracy de un clasificador trivial (siempre 0): {balance.loc[0, '%']:.2f} %")

df[TARGET].value_counts().sort_index().plot.bar(color=["tab:blue", "tab:red"])
plt.xticks([0, 1], ["No cancela", "Cancela"], rotation=0)
plt.title("Distribución de la variable objetivo")
plt.ylabel("Reservas")
plt.show()

## 3. Calidad de los datos

### 3.1 Valores nulos

In [ ]:
nulos = df.isna().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
pd.DataFrame({"nulos": nulos, "%": (nulos / len(df) * 100).round(2)})

`company` y `agent` tienen nulos con significado: indican que la reserva no se hizo a
través de una empresa o de un agente. No son datos perdidos al azar, así que conviene
comprobar si la ausencia está relacionada con la cancelación.

In [ ]:
for col in ["agent", "company"]:
    tasa = df.groupby(df[col].isna())[TARGET].mean().mul(100).round(2)
    tasa.index = [f"{col} informado", f"{col} nulo"]
    print(tasa, "\n")

### 3.2 Duplicados

In [ ]:
n_dup = df.duplicated().sum()
print(f"Filas duplicadas: {n_dup} ({n_dup / len(df):.2%})")
print("Tasa de cancelación en duplicados:", round(df[df.duplicated(keep=False)][TARGET].mean(), 4))
print("Tasa de cancelación global:       ", round(df[TARGET].mean(), 4))

Las filas idénticas pueden ser reservas de grupo legítimas, pero con un split aleatorio
una copia puede caer en train y otra en test, lo que infla las métricas.

### 3.3 Valores anómalos

In [ ]:
total_huespedes = df["adults"] + df["children"].fillna(0) + df["babies"]
total_noches = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]

anomalias = {
    "adr negativo": (df["adr"] < 0).sum(),
    "adr = 0": (df["adr"] == 0).sum(),
    "adr > 1000": (df["adr"] > 1000).sum(),
    "reservas sin huéspedes": (total_huespedes == 0).sum(),
    "reservas de 0 noches": (total_noches == 0).sum(),
}
pd.Series(anomalias, name="filas")

## 4. Fuga de información

`reservation_status` y `reservation_status_date` describen el desenlace de la reserva,
que solo se conoce después de que ocurra. Si se usan como predictores, el modelo lee la
respuesta en lugar de aprenderla.

In [ ]:
pd.crosstab(df["reservation_status"], df[TARGET])

In [ ]:
meses = {m: i for i, m in enumerate(
    ["January", "February", "March", "April", "May", "June", "July",
     "August", "September", "October", "November", "December"], start=1)}

fecha_llegada = pd.to_datetime(dict(
    year=df["arrival_date_year"],
    month=df["arrival_date_month"].map(meses),
    day=df["arrival_date_day_of_month"],
))
dias_estado = (pd.to_datetime(df["reservation_status_date"]) - fecha_llegada).dt.days

# Una fecha de estado anterior a la llegada delata una cancelación
pd.crosstab(dias_estado < 0, df[TARGET], rownames=["estado antes de la llegada"])

Ambas columnas separan el target casi a la perfección. Se eliminan antes de entrenar.

## 5. Variables numéricas

In [ ]:
LEAKAGE_COLS = ["reservation_status", "reservation_status_date"]
features = df.drop(columns=LEAKAGE_COLS + [TARGET])

num_cols = features.select_dtypes(include="number").columns.tolist()
cat_cols = features.select_dtypes(exclude="number").columns.tolist()
print(f"Numéricas ({len(num_cols)}):", num_cols)
print(f"Categóricas ({len(cat_cols)}):", cat_cols)

In [ ]:
df[num_cols].describe().T

In [ ]:
df[num_cols].hist(bins=40, figsize=(16, 14))
plt.suptitle("Distribución de las variables numéricas")
plt.tight_layout()
plt.show()

`agent` y `company` son numéricas en el CSV, pero son identificadores: su magnitud no
significa nada. Deben tratarse como categóricas.

Varias variables (`lead_time`, `adr`, `days_in_waiting_list`, `previous_cancellations`)
tienen colas largas, algo relevante para los modelos sensibles a la escala (regresión
logística, red neuronal) y poco relevante para los basados en árboles.

In [ ]:
cols_box = ["lead_time", "adr", "days_in_waiting_list", "booking_changes"]
fig, axes = plt.subplots(1, len(cols_box), figsize=(16, 4))
for ax, col in zip(axes, cols_box):
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("is_canceled")
plt.suptitle("Variables numéricas según cancelación")
plt.tight_layout()
plt.show()

## 6. Variables categóricas

La cardinalidad decide la codificación: one-hot es viable con pocas categorías, pero
genera cientos de columnas dispersas cuando hay muchas.

In [ ]:
cardinalidad = df[cat_cols + ["agent", "company"]].nunique().sort_values(ascending=False)
cardinalidad.to_frame("valores únicos")

In [ ]:
def tasa_cancelacion(col, top=None):
    """Devuelve el número de reservas y la tasa de cancelación por categoría."""
    tabla = (df.groupby(col)[TARGET]
               .agg(reservas="size", tasa_cancelacion="mean")
               .sort_values("reservas", ascending=False))
    tabla["tasa_cancelacion"] = tabla["tasa_cancelacion"].mul(100).round(2)
    return tabla.head(top) if top else tabla


cols_baja_card = ["hotel", "deposit_type", "customer_type", "market_segment",
                  "distribution_channel", "meal"]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), cols_baja_card):
    tasa_cancelacion(col)["tasa_cancelacion"].plot.bar(ax=ax, color="tab:red")
    ax.axhline(df[TARGET].mean() * 100, color="k", linestyle="--", linewidth=1)
    ax.set_title(col)
    ax.set_ylabel("% cancelación")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
plt.suptitle("Tasa de cancelación por categoría (línea: media global)")
plt.tight_layout()
plt.show()

In [ ]:
tasa_cancelacion("deposit_type")

In [ ]:
tasa_cancelacion("country", top=15)

## 7. Relación con la cancelación

### 7.1 Correlación de las numéricas con el target

In [ ]:
num_reales = [c for c in num_cols if c not in ("agent", "company")]
corr_target = df[num_reales + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()

corr_target.plot.barh(figsize=(8, 7), color=np.where(corr_target > 0, "tab:red", "tab:blue"))
plt.title("Correlación de Pearson con is_canceled")
plt.axvline(0, color="k", linewidth=0.8)
plt.show()

In [ ]:
corr = df[num_reales].corr()
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90)
ax.set_yticks(range(len(corr)), corr.columns)
fig.colorbar(im)
ax.set_title("Matriz de correlación entre variables numéricas")
plt.tight_layout()
plt.show()

### 7.2 Antelación de la reserva (`lead_time`)

In [ ]:
tramos = pd.cut(df["lead_time"], bins=[-1, 7, 30, 90, 180, 365, df["lead_time"].max()])
df.groupby(tramos, observed=True)[TARGET].mean().mul(100).plot.bar(color="tab:red")
plt.title("Tasa de cancelación por antelación (días)")
plt.ylabel("% cancelación")
plt.xticks(rotation=0)
plt.show()

### 7.3 Variables que separan el target de forma sospechosa

Una variable que separa las clases a la perfección puede ser una fuga de información
encubierta: un dato que solo se registra después del momento de predicción.

In [ ]:
tasa_cancelacion("required_car_parking_spaces")

## 8. Evolución temporal

In [ ]:
mensual = (df.assign(fecha=fecha_llegada.dt.to_period("M"))
             .groupby("fecha")[TARGET]
             .agg(reservas="size", tasa_cancelacion="mean"))

fig, ax1 = plt.subplots(figsize=(14, 4))
mensual["reservas"].plot.bar(ax=ax1, color="lightgray")
ax1.set_ylabel("Reservas")
ax2 = ax1.twinx()
ax2.plot(range(len(mensual)), mensual["tasa_cancelacion"] * 100, color="tab:red", marker="o")
ax2.set_ylabel("% cancelación")
ax1.set_title("Reservas y tasa de cancelación por mes de llegada")
plt.tight_layout()
plt.show()

In [ ]:
df.groupby(["arrival_date_year", "hotel"])[TARGET].mean().mul(100).round(2).unstack()

Los datos cubren un periodo de tiempo, así que un split aleatorio mezcla reservas
futuras en el entrenamiento. Una validación temporal (entrenar con los primeros años y
evaluar con el último) da una estimación más realista del rendimiento en producción.

## 9. Conclusiones para el modelado

| Hallazgo | Decisión |
| --- | --- |
| `reservation_status` y `reservation_status_date` delatan el target | Eliminarlas antes de entrenar |
| Clases desbalanceadas | Métrica principal F1-Score (ROC-AUC como secundaria); partición estratificada |
| Nulos en `agent`, `company`, `children`, `country` | Imputación dentro del `Pipeline` para no filtrar información del test |
| `agent` y `company` son identificadores | Tratarlas como categóricas, no como numéricas |
| Alta cardinalidad en `country`, `agent`, `company` | One-hot para modelos lineales y red neuronal; codificación ordinal para árboles |
| Colas largas en `lead_time`, `adr` y otras | Escalado para modelos lineales y red neuronal |
| Muchas filas duplicadas | Tenerlo en cuenta al interpretar las métricas del split aleatorio |
| `required_car_parking_spaces` separa el target de forma sospechosa | Valorar su eliminación o documentar el riesgo |
| `deposit_type` y `lead_time` muy asociadas a la cancelación | Predictores fuertes y legítimos |
| Tendencia temporal en la tasa de cancelación | Proponer validación temporal como mejora |